# Decisão de campanha: conversão com limite de risco

**Pergunta:** adotar a variante B aumenta conversão sem elevar descadastros além do tolerado?
Este é um experimento sintético, não uma campanha executada. O objetivo é demonstrar desenho,
análise e uma decisão condicionada à incerteza. Execute todas as células em ordem.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / 'src/experiment.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src/experiment.py').exists():
    raise RuntimeError('Abra o notebook na raiz do repositório ou em notebooks/.')
sys.path.insert(0, str(ROOT / 'src'))
from experiment import plan, validate, analyze, peeking_simulation


## 1. Plano antes da análise

Randomização por usuário, divisão 50/50 e análise por atribuição. O MDE é um alvo de desenho,
não uma garantia de efeito. A regra implementada exige estimativa ≥ MDE e IC excluindo zero;
ela não demonstra que o efeito verdadeiro excede o MDE. O guardrail exige que o limite superior
do IC do aumento de descadastros fique abaixo da tolerância. Um teste real requer registro
deste plano antes da coleta; a ordem das células não comprova pré-registro.

In [ ]:
specification = plan()
display(pd.Series(specification, name='Plano'))

## 2. Dados e checagens

Uma linha por usuário, sem exclusões pós-resultado. A base incluída foi gerada com semente fixa.
A duração de 28 dias é uma hipótese da simulação; a tabela não permite auditá-la por timestamps.

In [ ]:
frame = pd.read_csv(ROOT / 'data/experimento_sintetico.csv')
validate(frame)
display(frame.groupby('arm').agg(users=('user_id', 'size'), conversion=('converted', 'mean'), revenue_per_user=('revenue', 'mean'), unsubscribed=('unsubscribed', 'mean')))
result = analyze(frame, observation_days=28)
result['synthetic_data'] = True
display(pd.Series(result['validity'], name='Validade'))
display(pd.Series(result['srm'], name='SRM'))

## 3. Efeito primário e guardrail

O efeito absoluto e o IC estão em pontos percentuais. Significância de conversão não substitui
a avaliação de segurança. O intervalo para diferença de proporções usa Wilson/Newcombe.

In [ ]:
rows = []
for label, key in [('Conversão', 'primary_conversion'), ('Descadastro', 'guardrail_unsubscribed')]:
    m = result[key]
    rows.append({'métrica': label, 'A (%)': 100*m['rate_A'], 'B (%)': 100*m['rate_B'], 'efeito (p.p.)': 100*m['difference_absolute'], 'IC95 inferior (p.p.)': 100*m['ci95'][0], 'IC95 superior (p.p.)': 100*m['ci95'][1]})
display(pd.DataFrame(rows).set_index('métrica'))

## 4. Receita e sensibilidade

Receita por usuário é secundária. Compare Welch, bootstrap e ajuste por receita pré-teste
(ANCOVA com erro robusto HC3). O ajuste pressupõe uma covariável medida antes da intervenção;
não use informação pós-tratamento. Não selecione retrospectivamente a métrica que favorece B.

In [ ]:
display(pd.Series(result['secondary_revenue'], name='Welch'))
display(pd.Series(result['revenue_bootstrap'], name='Bootstrap'))
display(pd.Series(result['pretest_covariate_adjustment'], name='Ajuste pré-teste'))
display(pd.Series(result['secondary_holm_adjusted_p'], name='p ajustado Holm'))

## 5. Decisão e próximo passo

A conclusão abaixo é calculada com a mesma regra usada nos testes. Um IC de guardrail que
ultrapassa o limite não prova dano acima do limite: mostra que a segurança ainda não foi demonstrada.

In [ ]:
display(Markdown('**Decisão:** ' + result['decision']))
if not all(result['validity'].values()):
    next_step = 'Investigar validade e completar o plano antes de interpretar efeitos.'
elif result['decision'].startswith('CANDIDATO'):
    next_step = 'Avaliar custo e aplicação gradual com monitoramento; não extrapolar a simulação para clientes reais.'
elif result['decision'].startswith('REVISAR'):
    next_step = 'Investigar descadastros e planejar um novo teste; não prolongar este teste só até obter significância.'
else:
    next_step = 'Manter A e revisar a hipótese; um novo teste deve ter plano próprio.'
display(Markdown('**Próximo passo:** ' + next_step))

## 6. Por que não olhar o p-valor todo dia?

Experimento separado sob hipótese nula: compare uma única análise final com a regra de parar
quando qualquer uma de dez inspeções cruza 5%. A taxa observada é Monte Carlo, não uma constante
universal. Esta simulação não fornece uma correção sequencial para o teste principal.

In [ ]:
peeking = peeking_simulation(repetitions=1000)
display(peeking)

## Limites e conclusão de negócio

A decisão vale para esta simulação e suas hipóteses. Não há custos de campanha, receita líquida,
interferência entre usuários, sazonalidade ou auditoria de randomização real. Em aplicação real,
defina população elegível, janela de conversão, tratamento de perdas e política de risco antes
de coletar dados. O sucesso do projeto é uma decisão rastreável, inclusive quando a resposta é
não adotar ou reunir mais evidência em um novo estudo.